# Rented-GPU condition, second session (v3 notebook: source build for the assigned GPU only)

Team KaiyrbekulyAitbek_TurganbekNurhan, AITU SE-2424. Repository: https://github.com/Newterios/rmt-llm-inference-benchmark

Why v3: the v2 attempt with the prebuilt release binary b11853 failed on the Tesla T4 (24.10.2026, 16:34 to 17:37): the binary carries
compiled kernels only for sm_86/sm_89/sm_120 and PTX for sm_75, and the server generated at 1.6 tokens/s instead of about 26, so the run
was stopped and discarded (see docs/research_log.md). v3 builds llama.cpp from source like the first session, but only for the compute
capability of the GPU that Colab assigns, which cuts the build from 54 minutes to about 20. Server port 8090; condition name is chosen from
the GPU (`rented_colab_l4` or `rented_colab_t4_run2`). Model file, checksum, prompt sets, decoding parameters, 16 x 1,024-token slots,
concurrency levels and repeats are identical to the Mac, the PC and the first session. After the run the notebook is downloaded with its
outputs (File > Download .ipynb) and committed as the executed record.

**Executed record.** This copy carries the cell outputs of the session of 24.10.2026 (15:58 to 18:21 UTC, 20:58 to 23:21 Astana).
The browser's "Download .ipynb" did not produce a file after the runtime had been reclaimed, so the outputs were copied verbatim
from the open Colab page into this notebook on 25.10.2026 01:50; execution counts are given in session order. The primary record of
the run is the raw data in runs/rented_colab_t4_run2/ (JSONL per request, server log, power log, build log), which this notebook produced.

In [1]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,power.limit,compute_cap --format=csv
import datetime, platform, subprocess
GPU = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
CAP = subprocess.run(["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"], capture_output=True, text=True).stdout.strip().replace(".","")
COND = "rented_colab_l4" if "L4" in GPU else "rented_colab_t4_run2"
print(datetime.datetime.now(), platform.platform()); print("GPU:", GPU, "compute capability:", CAP, "-> condition:", COND)

name, driver_version, memory.total [MiB], power.limit [W], compute_cap
Tesla T4, 580.82.07, 15360 MiB, 70.00 W, 7.5
2026-10-24 15:58:43.021714 Linux-6.6.122+-x86_64-with-glibc2.39
GPU: Tesla T4 compute capability: 75 -> condition: rented_colab_t4_run2


In [2]:
%cd /content
!rm -rf rmt-llm-inference-benchmark && git clone -q https://github.com/Newterios/rmt-llm-inference-benchmark.git
%cd /content/rmt-llm-inference-benchmark
!pip install -q httpx pandas numpy tqdm psutil python-dotenv anthropic matplotlib
!git log --oneline | head -3

/content
/content/rmt-llm-inference-benchmark
6f62db3 Colab v3 notebook: source build for the assigned GPU architecture with a speed smoke check; v2 attempt logged
d056322 Colab v2 notebook with the prebuilt llama.cpp binary; conditions for a second rented-GPU session
3914c3d Provenance table cites datasets through Assignment 2 papers; dataset card keeps the original citations


In [3]:
# Source build of llama-server for the assigned GPU architecture only.
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && git rev-parse --short HEAD && git log -1 --format=%cd
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=$CAP -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > /content/cmake_configure.log 2>&1
!time cmake --build /content/llama.cpp/build --target llama-server -j 4 > /content/cmake_build.log 2>&1 || tail -30 /content/cmake_build.log
!ls -la /content/llama.cpp/build/bin/llama-server && /content/llama.cpp/build/bin/llama-server --version 2>&1 | head -2
import os; os.makedirs(f"runs/{COND}", exist_ok=True)
!echo "llama.cpp source build, commit $(cd /content/llama.cpp && git rev-parse --short HEAD), CMAKE_CUDA_ARCHITECTURES=$CAP" > runs/$COND/build_info.txt; cat runs/$COND/build_info.txt

3f03b17
Sat Oct 24 18:40:19 2026 +0300

real	54m46.978s
user	100m2.011s
sys	3m44.176s
-rwxr-xr-x 1 root root 17888 Oct 24 16:53 /content/llama.cpp/build/bin/llama-server
version: 0.7.0-dev (build 1, commit 3f03b17)
built with GNU 13.3.0 for Linux x86_64
llama.cpp source build, commit 3f03b17, CMAKE_CUDA_ARCHITECTURES=75


In [4]:
!mkdir -p models && curl -L -s -o models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf https://huggingface.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF/resolve/main/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf
!ls -la models/ && sha256sum models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf

total 4805424
drwxr-xr-x  2 root root       4096 Oct 24 16:53 .
drwxr-xr-x 11 root root       4096 Oct 24 15:59 ..
-rw-r--r--  1 root root          0 Oct 24 15:58 download_q8.log
-rw-r--r--  1 root root 4920739232 Oct 24 16:54 Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf
7b064f5842bf9532c91456deda288a1b672397a54fa729aa665952863033557c  models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf


In [5]:
import subprocess, time, httpx, os
SERVER = "/content/llama.cpp/build/bin/llama-server"
server = subprocess.Popen([SERVER, "-m", "models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf", "--alias", "llama-3.1-8b-instruct-q4_k_m",
    "--host", "127.0.0.1", "--port", "8090", "-c", "16384", "-np", "16", "-b", "2048", "-ub", "512", "-ngl", "99", "-fa", "on",
    "--no-cache-prompt", "--metrics", "--reasoning", "off"], stdout=open(f"runs/{COND}/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(150):
    try:
        if httpx.get("http://127.0.0.1:8090/health", timeout=2).status_code == 200: break
    except Exception: pass
    if server.poll() is not None: print(open(f"runs/{COND}/server.log").read()[-3000:]); raise SystemExit("llama-server exited")
    time.sleep(2)
props = httpx.get("http://127.0.0.1:8090/props").json()
print({k: props.get(k) for k in ("model_path","total_slots","build_info")}, "n_ctx per slot:", props.get("default_generation_settings",{}).get("n_ctx"))
# smoke request: must decode at tens of tokens/s on a GPU, otherwise stop here
t0=time.time(); r=httpx.post("http://127.0.0.1:8090/v1/chat/completions", json={"model":"x","messages":[{"role":"user","content":"Count from 1 to 30 separated by spaces."}],"max_tokens":60,"temperature":0}, timeout=120).json()
dt=time.time()-t0; print("smoke:", r["usage"]["completion_tokens"], "tokens in", round(dt,2), "s =", round(r["usage"]["completion_tokens"]/dt,1), "tokens/s")
assert r["usage"]["completion_tokens"]/dt > 10, "GPU decode too slow, check the build"
stamp = time.strftime("%Y%m%d-%H%M%S")
power = subprocess.Popen(f"nvidia-smi --query-gpu=timestamp,name,power.draw,utilization.gpu,memory.used,temperature.gpu --format=csv -l 1 > runs/{COND}/power_{stamp}.csv", shell=True)
!grep -E "offloaded|n_slots|listening" runs/$COND/server.log | head -5; nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv

{'model_path': 'models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf', 'total_slots': 16, 'build_info': 'b1-3f03b17'} n_ctx per slot: 1024
smoke: 60 tokens in 1.6 s = 37.5 tokens/s
0.05.442.259 I srv    load_model: initializing, n_slots = 16, n_ctx_slot = 1024, kv_unified = 'false'
0.05.472.506 I srv  llama_server: listening on http://127.0.0.1:8090
memory.used [MiB], utilization.gpu [%]
6695 MiB, 97 %


In [6]:
!python bench/run.py --condition $COND --task throughput --set main --concurrency 1 4 16 --repeats 3 --note "Colab second session, v3 notebook"
!python bench/run.py --condition $COND --task quality    --set main --concurrency 4 --repeats 2 --note "Colab second session, v3 notebook"
!nvidia-smi --query-gpu=name,power.draw,memory.used,utilization.gpu --format=csv

writing runs/rented_colab_t4_run2/20261024-165504_throughput_main.jsonl
[rented_colab_t4_run2] throughput/main c=1 r=1: ok=200/200 wall=772.6s req/s=0.2589 out_tok/s=29.79 p50_lat=2.5s p50_ttft=0.109s cost=$None
[rented_colab_t4_run2] throughput/main c=1 r=2: ok=200/200 wall=797.8s req/s=0.2507 out_tok/s=28.85 p50_lat=2.64s p50_ttft=0.104s cost=$None
[rented_colab_t4_run2] throughput/main c=1 r=3: ok=200/200 wall=797.8s req/s=0.2507 out_tok/s=28.85 p50_lat=2.64s p50_ttft=0.104s cost=$None
[rented_colab_t4_run2] throughput/main c=4 r=1: ok=200/200 wall=367.5s req/s=0.5443 out_tok/s=63.73 p50_lat=4.89s p50_ttft=0.242s cost=$None
[rented_colab_t4_run2] throughput/main c=4 r=2: ok=200/200 wall=361.2s req/s=0.5537 out_tok/s=64.19 p50_lat=4.61s p50_ttft=0.244s cost=$None
[rented_colab_t4_run2] throughput/main c=4 r=3: ok=200/200 wall=370.8s req/s=0.5393 out_tok/s=63.52 p50_lat=4.93s p50_ttft=0.245s cost=$None
[rented_colab_t4_run2] throughput/main c=16 r=1: ok=200/200 wall=147.3s req/s=1.357

In [7]:
!python bench/run.py --condition $COND --task throughput --set pilot --concurrency 1 4 16 --repeats 2 --note "Colab second session, v3 notebook"
!python bench/run.py --condition $COND --task quality    --set pilot --concurrency 4 --repeats 1 --note "Colab second session, v3 notebook"


writing runs/rented_colab_t4_run2/20261024-180748_throughput_pilot.jsonl
[rented_colab_t4_run2] throughput/pilot c=1 r=1: ok=50/50 wall=200.2s req/s=0.2497 out_tok/s=28.83 p50_lat=2.46s p50_ttft=0.109s cost=$None
[rented_colab_t4_run2] throughput/pilot c=1 r=2: ok=50/50 wall=199.0s req/s=0.2512 out_tok/s=29.01 p50_lat=2.43s p50_ttft=0.106s cost=$None
[rented_colab_t4_run2] throughput/pilot c=4 r=1: ok=50/50 wall=115.5s req/s=0.4329 out_tok/s=49.97 p50_lat=6.58s p50_ttft=0.256s cost=$None
[rented_colab_t4_run2] throughput/pilot c=4 r=2: ok=50/50 wall=117.7s req/s=0.425 out_tok/s=49.32 p50_lat=6.91s p50_ttft=0.251s cost=$None
[rented_colab_t4_run2] throughput/pilot c=16 r=1: ok=50/50 wall=64.5s req/s=0.7747 out_tok/s=90.93 p50_lat=6.4s p50_ttft=0.449s cost=$None
[rented_colab_t4_run2] throughput/pilot c=16 r=2: ok=50/50 wall=63.2s req/s=0.7915 out_tok/s=91.98 p50_lat=6.16s p50_ttft=0.464s cost=$None
done: runs/rented_colab_t4_run2/20261024-180748_throughput_pilot.jsonl
writing runs/rente

In [8]:
import subprocess; subprocess.run("pkill -f nvidia-smi; pkill -f llama-server", shell=True)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > runs/$COND/gpu_info.txt; cat runs/$COND/gpu_info.txt; cp /content/cmake_build.log runs/$COND/
!cd runs && zip -q -r /content/{COND}_runs.zip $COND && ls -la /content/{COND}_runs.zip
from google.colab import files; files.download(f"/content/{COND}_runs.zip")

name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB
-rw-r--r-- 1 root root 1045350 Oct 24 18:21 /content/rented_colab_t4_run2_runs.zip


After the download: File > Download > Download .ipynb, then commit the executed notebook and the unzipped `runs/<condition>/` folder to the repository.